# Rainfall Pattern Analysis and Prediction System
## Historical Observation Analysis & Machine Learning Modeling

This notebook demonstrates:
1. Loading and preprocessing the India Meteorological Department (IMD) district-wise daily rainfall dataset.
2. Historical rainfall analysis for key districts (e.g., Pune, Mumbai City, Nagpur).
3. Chronological train-test splitting and feature extraction.
4. Training Baseline, Logistic Regression, and Random Forest Classifier pipelines.
5. Evaluating models using Accuracy, Precision, Recall, F1, ROC-AUC, and Brier Score.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Ensure project root is in sys.path
sys.path.insert(0, '..')

from src.data_processing import load_and_preprocess_data, calculate_historical_statistics, filter_by_location, get_dataset_metadata
from src.train_model import train_and_save_models
from src.prediction import load_model_bundle, estimate_rainfall_probability

### 1. Dataset Inspection & Preprocessing

In [ ]:
df = load_and_preprocess_data('../data/rainfall_districtwise_daily_imd.csv')
print(f"Total records: {len(df):,}")
print(f"Unique districts: {df['district'].nunique()}")
print(f"Unique states: {df['state'].nunique()}")
print(f"Observation window: {df['date'].min().strftime('%d-%b-%Y')} to {df['date'].max().strftime('%d-%b-%Y')}")
df.head()

### 2. District-Level Analysis (Pune, Mumbai City, Nagpur)

In [ ]:
for dist in ['PUNE', 'MUMBAI CITY', 'NAGPUR']:
    dist_df = filter_by_location(df, dist)
    stats = calculate_historical_statistics(dist_df)
    print(f"=== {dist} ({stats['state']}) ===")
    print(f"Total Rainfall: {stats['total_rainfall_mm']:.1f} mm over {stats['total_days']} days")
    print(f"Rainy Days (>= 2.5 mm): {stats['rainy_days_count']} ({stats['rainy_percentage']:.1f}%)")
    print(f"Daily Average: {stats['avg_daily_rainfall_mm']:.2f} mm/day")
    print(f"Peak Single Day: {stats['max_single_day_rain_mm']:.1f} mm on {stats['max_single_day_date']}\n")

### 3. Model Training & Chronological Evaluation

In [ ]:
bundle = train_and_save_models(
    data_path='../data/rainfall_districtwise_daily_imd.csv',
    model_output_path='../models/rainfall_model.pkl'
)

# Compare Model Evaluation Metrics
metrics_df = pd.DataFrame(bundle['metrics']).T
metrics_df

### 4. Estimating Rainfall Probabilities by Month

In [ ]:
target_dist = 'PUNE'
for m in [8, 9, 10]:
    prob_res = estimate_rainfall_probability(target_dist, month=m, df_all=df, model_bundle=bundle)
    print(f"Month {m} | Hist Freq: {prob_res['historical_frequency_pct']:.1f}% | RF Prob: {prob_res['rf_probability_pct']:.1f}% | LR Prob: {prob_res['lr_probability_pct']:.1f}%")
    print(f"Interpretation: {prob_res['interpretation']}\n")